# Project 1 - Milestone 4: Integrated detection + identification pipeline
**IE7615 - Group 02**

```
image -> YOLOv8 detector (Milestone 3) -> one box per face -> crop
      -> ResNet18 classifier (Milestone 1 architecture, retrained on all 13 identities)
      -> detector and classifier votes combined -> named boxes
```

The Milestone 1 classifier knew only Group 02's 4 identities, so it is retrained on all 13 class-pool
identities using the **same photo-level split** as the Milestone 3 detection dataset: no test photo is
seen by either model during training, and both models share class ids.

Runs on a laptop CPU in about 5-10 minutes (no GPU needed). The same steps run from the command line
with `python src/run_milestone4.py`.

In [ ]:
import os, sys
from pathlib import Path
here = Path.cwd().resolve()
ROOT = next(p for p in [here, *here.parents] if (p / "src" / "m4_pipeline.py").is_file())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import Image, Markdown, display
import m3_detection as m3, m4_pipeline as m4, run_milestone4 as pipeline
display(m4.split_counts())

## 1. Train the 13-identity classifier
ResNet18 pretrained on ImageNet (as in Milestone 1), fine-tuned with AdamW, 30 epochs, label smoothing 0.1, light augmentation that mimics detector crops. The checkpoint with the best validation accuracy is kept.

In [ ]:
SKIP_TRAINING = m4.CLASSIFIER_WEIGHTS.is_file()   # set False to retrain
hist = pipeline.step_train(epochs=30, smoke=False, skip=SKIP_TRAINING)
display(Image(str(m4.RESULTS_DIR / "classifier_curves.png")))

## 2. Classifier on single-face photos

In [ ]:
clf = m4.Classifier()
ev = pipeline.step_classifier_eval(clf)
display(m4.per_identity_accuracy(ev["test"]))
display(Image(str(m4.RESULTS_DIR / "classifier_confusion_test.png")))

## 3. Integrated pipeline on multi-face grids
Detector alone (Milestone 3) vs detector boxes named by the classifier vs both votes combined. The setting is **chosen on validation**; test is reported once.

In [ ]:
pipe = m4.Pipeline(classifier=clf)
val, test, CHOSEN = pipeline.step_pipeline(pipe)
display(val.round(3)); display(test.round(3)); print("chosen on validation:", CHOSEN)

## 4. Final results gallery
>=20 correct and >=5 misclassified single faces, >=10 multi-face images. Full captions: `results/milestone4/gallery/README.md`.

In [ ]:
counts = pipeline.step_gallery(clf, pipe, ev, CHOSEN)
print(counts)
import glob
for f in sorted(glob.glob(str(m4.RESULTS_DIR / "gallery" / "single_misclassified" / "*.jpg")))[:3]:
    display(Image(f, width=420))
for f in sorted(glob.glob(str(m4.RESULTS_DIR / "gallery" / "multi_face" / "*.jpg")))[:2]:
    display(Image(f, width=420))

## 5. Summary and live demo
For the live session: `python src/identify.py --app` (upload any image) or `python src/identify.py --source <image or folder>`.

In [ ]:
summary = pipeline.step_summary(hist, ev, val, test, CHOSEN, counts, clf)
display(Markdown((m4.RESULTS_DIR / "results_tables.md").read_text()))